# Lab 4 - Airbnb Team16 Spark DataFrame Analysis

This notebook analyzes Airbnb data with Spark Cluster on BigDis, following the sample Lab4.AirBnB_Analyis.ipynb.


## Cell 1 - Import Libraries and Create SparkSession


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    avg,
    desc,
    asc,
    regexp_replace,
    round,
    when,
    to_date,
    date_format,
    min as spark_min,
    max as spark_max
)

spark = SparkSession.builder \
    .appName("Lab4 Airbnb Team16 Spark DataFrame Analysis") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.range(10).show()



## Cell 2 - Configure Data and Output Paths


In [ ]:
import os

BASE_PATH = "/home/ubuntu/airbnb-spark-lab/data/raw"
OUTPUT_TABLE_PATH = "/home/ubuntu/airbnb-spark-lab/outputs/tables"
OUTPUT_CHART_PATH = "/home/ubuntu/airbnb-spark-lab/outputs/charts"

os.makedirs(OUTPUT_TABLE_PATH, exist_ok=True)
os.makedirs(OUTPUT_CHART_PATH, exist_ok=True)

print("Base path:", BASE_PATH)
print("Output table path:", OUTPUT_TABLE_PATH)
print("Output chart path:", OUTPUT_CHART_PATH)


## Cell 3 - Load the 4 Airbnb Datasets


In [ ]:
listings = spark.read.csv(
    f"{BASE_PATH}/listings.csv.gz",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

calendar = spark.read.csv(
    f"{BASE_PATH}/calendar.csv.gz",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

reviews = spark.read.csv(
    f"{BASE_PATH}/reviews.csv.gz",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

neighbourhoods = spark.read.csv(
    f"{BASE_PATH}/neighbourhoods.csv",
    header=True,
    inferSchema=True
)

print("Listings:", listings.count())
print("Calendar:", calendar.count())
print("Reviews:", reviews.count())
print("Neighbourhoods:", neighbourhoods.count())


## Cell 4 - Preview Sample Records


In [ ]:
print("Listings sample")
listings.show(5, truncate=False)

print("Calendar sample")
calendar.show(5, truncate=False)

print("Reviews sample")
reviews.show(5, truncate=False)

print("Neighbourhoods sample")
neighbourhoods.show(5, truncate=False)


## Cell 5 - Inspect Main Dataset Schemas


In [ ]:
print("Listings schema")
listings.printSchema()

print("Calendar schema")
calendar.printSchema()

print("Reviews schema")
reviews.printSchema()

print("Neighbourhoods schema")
neighbourhoods.printSchema()


**Comment:** The schema shows that `listings` is the main table, containing listing, host, location, room type, price, availability, and review metric fields. The `calendar`, `reviews`, and `neighbourhoods` tables support availability analysis, actual review counts, and neighbourhood-level analysis.


## Cell 6 - Check Dataset Geographic Coverage


In [ ]:
location_summary = listings.select(
    spark_min("latitude").alias("min_latitude"),
    spark_max("latitude").alias("max_latitude"),
    spark_min("longitude").alias("min_longitude"),
    spark_max("longitude").alias("max_longitude"),
    countDistinct("neighbourhood_cleansed").alias("distinct_neighbourhoods")
)

location_summary.show()

if "neighbourhood_group_cleansed" in listings.columns:
    listings.groupBy("neighbourhood_group_cleansed") \
        .agg(count("*").alias("listing_count")) \
        .orderBy(desc("listing_count")) \
        .show(20, truncate=False)
else:
    listings.groupBy("neighbourhood_cleansed") \
        .agg(count("*").alias("listing_count")) \
        .orderBy(desc("listing_count")) \
        .show(20, truncate=False)


**Comment:** This cell helps confirm that the dataset belongs to the correct city or assigned area. Inside Airbnb provides data by city, so latitude/longitude ranges and neighbourhood names should be checked before writing the final report.


## Cell 7 - Clean Listings Data and Price


In [ ]:
listings_clean = listings.withColumn(
    "price_clean",
    regexp_replace(col("price"), "[$,]", "").cast("double")
)

important_cols = [
    "id",
    "name",
    "host_id",
    "host_name",
    "neighbourhood_group_cleansed",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "price",
    "price_clean",
    "minimum_nights",
    "maximum_nights",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating",
    "calculated_host_listings_count"
]

existing_cols = [c for c in important_cols if c in listings_clean.columns]
listings_clean = listings_clean.select(*existing_cols)

listings_clean.show(5, truncate=False)


## Cell 8 - Missing Value / Null Report


In [ ]:
null_report = []

for c in listings_clean.columns:
    null_count = listings_clean.where(col(c).isNull()).count()
    null_report.append((c, null_count))

null_df = spark.createDataFrame(null_report, ["column_name", "null_count"])
null_df.orderBy(desc("null_count")).show(30)


**Comment:** The null report identifies columns with missing data. For price-related analysis, records without `price_clean` are removed to avoid misleading results.


## Cell 9 - Filter Listings with Valid Price


In [ ]:
listings_clean = listings_clean.where(col("price_clean").isNotNull())

print("Listings after removing missing price:", listings_clean.count())


## Cell 10 - Select the Best Region Column


In [ ]:
if "neighbourhood_group_cleansed" in listings_clean.columns and listings_clean.filter(col("neighbourhood_group_cleansed").isNotNull()).limit(1).count() > 0:
    region_col = "neighbourhood_group_cleansed"
else:
    region_col = "neighbourhood_cleansed"

print("Using region column:", region_col)


## Cell 11 - Hosts with the Most Listings


In [ ]:
host_listing_count = listings_clean.groupBy("host_id", "host_name") \
    .agg(count("*").alias("listing_count")) \
    .orderBy(desc("listing_count"))

host_listing_count.show(20, truncate=False)

if "calculated_host_listings_count" in listings_clean.columns:
    listings_clean.select(
        spark_max("calculated_host_listings_count").alias("max_calculated_host_listings_count")
    ).show()


**Comment:** This analysis identifies hosts with the largest number of listings. A high listing count may indicate professional hosts or property management companies.


## Cell 12 - Listing Count by Region


In [ ]:
region_count = listings_clean.groupBy(region_col) \
    .agg(count("*").alias("listing_count")) \
    .orderBy(desc("listing_count"))

region_count.show(30, truncate=False)


**Comment:** The listing distribution by region shows which areas have larger Airbnb supply, often related to tourism, central locations, or convenient transportation.


## Cell 13 - Chart: Listings by Region


In [ ]:
import matplotlib.pyplot as plt

region_pd = region_count.toPandas()

plt.figure(figsize=(8, 8))
plt.pie(
    region_pd["listing_count"],
    labels=region_pd[region_col],
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Listings by Region")
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/listings_by_region.png")
plt.show()


## Cell 14 - Listing Count by Room Type


In [ ]:
room_type_count = listings_clean.groupBy("room_type") \
    .agg(count("*").alias("listing_count")) \
    .orderBy(desc("listing_count"))

room_type_count.show()


**Comment:** The room type distribution shows the structure of Airbnb supply. A high proportion of `Entire home/apt` listings may indicate a stronger short-term rental market.


## Cell 15 - Chart: Listings by Room Type


In [ ]:
room_pd = room_type_count.toPandas()

plt.figure(figsize=(8, 8))
plt.pie(
    room_pd["listing_count"],
    labels=room_pd["room_type"],
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Listings by Room Type")
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/listings_by_room_type.png")
plt.show()


## Cell 16 - Room Type by Region


In [ ]:
room_by_region = listings_clean.groupBy(region_col, "room_type") \
    .agg(count("*").alias("listing_count")) \
    .orderBy(region_col, desc("listing_count"))

room_by_region.show(50, truncate=False)


**Comment:** This analysis compares room type composition across regions and helps identify whether each area is dominated by entire homes, private rooms, hotel rooms, or shared rooms.


## Cell 17 - Chart: Room Type by Region


In [ ]:
room_region_pd = room_by_region.toPandas()

pivot_room_region = room_region_pd.pivot(
    index=region_col,
    columns="room_type",
    values="listing_count"
).fillna(0)

pivot_room_region.plot(kind="bar", figsize=(12, 6))
plt.title("Room Type Distribution by Region")
plt.xlabel("Region")
plt.ylabel("Listing Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/room_type_by_region.png")
plt.show()


## Cell 18 - Average Price by Region and Room Type


In [ ]:
avg_price_by_region_room = listings_clean.groupBy(region_col, "room_type") \
    .agg(
        count("*").alias("listing_count"),
        round(avg("price_clean"), 2).alias("avg_price")
    ) \
    .orderBy(region_col, desc("avg_price"))

avg_price_by_region_room.show(50, truncate=False)


**Comment:** Average price analysis compares price levels across regions and room types. Entire homes or apartments usually have higher average prices than private or shared rooms.


## Cell 19 - Chart: Average Price by Region and Room Type


In [ ]:
price_region_pd = avg_price_by_region_room.toPandas()

pivot_price_region = price_region_pd.pivot(
    index=region_col,
    columns="room_type",
    values="avg_price"
).fillna(0)

pivot_price_region.plot(kind="bar", figsize=(12, 6))
plt.title("Average Price by Region and Room Type")
plt.xlabel("Region")
plt.ylabel("Average Price")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/avg_price_by_region_room_type.png")
plt.show()


## Cell 20 - Top 10 Neighbourhoods by Listing Count


In [ ]:
top_neighbourhoods = listings_clean.groupBy("neighbourhood_cleansed") \
    .agg(count("*").alias("listing_count")) \
    .orderBy(desc("listing_count")) \
    .limit(10)

top_neighbourhoods.show(truncate=False)


**Comment:** The top neighbourhood analysis identifies areas with the highest number of Airbnb listings. These are likely important accommodation zones near tourist attractions, business districts, or transport hubs.


## Cell 21 - Chart: Top 10 Neighbourhoods


In [ ]:
top_neighbourhoods_pd = top_neighbourhoods.toPandas()

plt.figure(figsize=(10, 6))
plt.barh(
    top_neighbourhoods_pd["neighbourhood_cleansed"],
    top_neighbourhoods_pd["listing_count"]
)
plt.title("Top 10 Neighbourhoods by Listing Count")
plt.xlabel("Listing Count")
plt.ylabel("Neighbourhood")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/top_neighbourhoods.png")
plt.show()


## Cell 22 - Minimum Nights Distribution


In [ ]:
minimum_nights_dist = listings_clean.groupBy("minimum_nights") \
    .agg(count("*").alias("listing_count")) \
    .orderBy(asc("minimum_nights"))

minimum_nights_dist.show(30)


**Comment:** The minimum nights distribution reflects booking policy patterns. Very large values should be interpreted as outliers or long-term rental policies.


## Cell 23 - Chart: Minimum Nights from 1 to 30


In [ ]:
min_nights_30 = minimum_nights_dist.where(col("minimum_nights") <= 30)
min_nights_pd = min_nights_30.toPandas()

plt.figure(figsize=(12, 6))
plt.bar(min_nights_pd["minimum_nights"], min_nights_pd["listing_count"])
plt.title("Distribution of Minimum Nights")
plt.xlabel("Minimum Nights")
plt.ylabel("Listing Count")
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/minimum_nights_distribution.png")
plt.show()


## Cell 24 - Popular Listings by Number of Reviews


In [ ]:
popular_listings = listings_clean.select(
    "id",
    "name",
    "host_name",
    "neighbourhood_cleansed",
    "room_type",
    "price_clean",
    "number_of_reviews",
    "review_scores_rating"
).orderBy(desc("number_of_reviews"))

popular_listings.show(20, truncate=False)


**Comment:** Listings with many reviews can be considered active or popular listings. However, review count does not directly measure quality because it also depends on listing age and booking frequency.


## Cell 25 - Clean Calendar Dataset


In [ ]:
calendar_clean = calendar.withColumn(
    "calendar_price_clean",
    regexp_replace(col("price"), "[$,]", "").cast("double")
).withColumn(
    "is_available",
    when(col("available") == "t", 1).otherwise(0)
).withColumn(
    "date",
    to_date(col("date"))
)

calendar_clean.select(
    "listing_id", "date", "available", "is_available", "calendar_price_clean"
).show(10)


## Cell 26 - Calendar Date Range


In [ ]:
calendar_clean.select(
    spark_min("date").alias("start_date"),
    spark_max("date").alias("end_date"),
    countDistinct("date").alias("number_of_dates")
).show()


**Comment:** The date range shows the period covered by the calendar table and helps explain the time window used in the availability chart.


## Cell 27 - Daily Calendar Availability


In [ ]:
daily_availability = calendar_clean.groupBy("date") \
    .agg(
        count("*").alias("calendar_records"),
        round(avg("is_available"), 4).alias("availability_rate"),
        round(avg("calendar_price_clean"), 2).alias("avg_calendar_price")
    ) \
    .orderBy("date")

daily_availability.show(20)


**Comment:** Availability rate measures the proportion of listing-date records that are available for booking. Average calendar price shows daily price movement in the calendar data.


## Cell 28 - Chart: Daily Availability Rate


In [ ]:
daily_availability_pd = daily_availability.limit(60).toPandas()

plt.figure(figsize=(12, 6))
plt.plot(
    daily_availability_pd["date"],
    daily_availability_pd["availability_rate"]
)
plt.title("Daily Availability Rate")
plt.xlabel("Date")
plt.ylabel("Availability Rate")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/daily_availability_rate.png")
plt.show()


## Cell 29 - Clean Reviews Dataset


In [ ]:
reviews_clean = reviews.withColumn(
    "date",
    to_date(col("date"))
)

reviews_clean.show(5, truncate=False)


## Cell 30 - Actual Review Count by Listing


In [ ]:
review_count_by_listing = reviews_clean.groupBy("listing_id") \
    .agg(count("*").alias("actual_review_count"))

review_count_by_listing.show(10)


## Cell 31 - Join Listings with Reviews


In [ ]:
listing_review_join = listings_clean.join(
    review_count_by_listing,
    listings_clean.id == review_count_by_listing.listing_id,
    "left"
).select(
    listings_clean.id,
    "name",
    "room_type",
    "neighbourhood_cleansed",
    "price_clean",
    "number_of_reviews",
    "actual_review_count"
)

listing_review_join.orderBy(desc("actual_review_count")).show(20, truncate=False)


**Comment:** Joining listings with reviews allows comparison between `number_of_reviews` stored in listings and the actual number of review records calculated from the reviews table.


## Cell 32 - Scatter Map by Latitude, Longitude, and Price


In [ ]:
geo_df = listings_clean.select(
    "latitude",
    "longitude",
    "price_clean",
    region_col,
    "room_type"
).where(
    col("latitude").isNotNull() &
    col("longitude").isNotNull() &
    col("price_clean").isNotNull()
)

geo_df_filtered = geo_df.where(col("price_clean") <= 500)
geo_pd = geo_df_filtered.toPandas()

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    geo_pd["longitude"],
    geo_pd["latitude"],
    c=geo_pd["price_clean"],
    s=8,
    alpha=0.7
)
plt.colorbar(scatter, label="Price")
plt.title("Airbnb Listings Distribution by Location and Price")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/listing_location_price_scatter.png")
plt.show()


**Comment:** The scatter plot visualizes listing distribution by longitude and latitude, with color representing price. It helps identify dense listing areas and price differences by location.


## Cell 33 - Price Distribution


In [ ]:
price_summary = listings_clean.agg(
    count("*").alias("listing_count"),
    round(avg("price_clean"), 2).alias("avg_price"),
    spark_min("price_clean").alias("min_price"),
    spark_max("price_clean").alias("max_price")
)

price_summary.show()

price_quantiles = listings_clean.approxQuantile(
    "price_clean",
    [0.25, 0.50, 0.75, 0.90, 0.95],
    0.01
)

price_quantile_summary = spark.createDataFrame(
    [
        ("q25", float(price_quantiles[0])),
        ("q50_median", float(price_quantiles[1])),
        ("q75", float(price_quantiles[2])),
        ("q90", float(price_quantiles[3])),
        ("q95", float(price_quantiles[4]))
    ],
    ["metric", "value"]
)

price_quantile_summary.show()



**Comment:** Price distribution helps identify the typical price range and extreme outliers. The histogram below filters prices at 500 to make the main distribution easier to read.


## Cell 34 - Chart: Price Distribution


In [ ]:
price_dist_pd = listings_clean.where(col("price_clean") <= 500) \
    .select("price_clean") \
    .toPandas()

plt.figure(figsize=(10, 6))
plt.hist(price_dist_pd["price_clean"], bins=40, edgecolor="black")
plt.title("Price Distribution for Listings with Price <= 500")
plt.xlabel("Price")
plt.ylabel("Listing Count")
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/price_distribution.png")
plt.show()


## Cell 35 - Review Score by Room Type


In [ ]:
review_score_by_room_type = listings_clean.where(col("review_scores_rating").isNotNull()) \
    .groupBy("room_type") \
    .agg(
        count("*").alias("rated_listing_count"),
        round(avg("review_scores_rating"), 2).alias("avg_review_score")
    ) \
    .orderBy(desc("avg_review_score"))

review_score_by_room_type.show(truncate=False)


**Comment:** Review score by room type compares guest satisfaction across different accommodation types. It adds a quality-oriented view beyond listing count and price.


## Cell 36 - Chart: Review Score by Room Type


In [ ]:
review_score_room_pd = review_score_by_room_type.toPandas()

plt.figure(figsize=(8, 6))
plt.bar(
    review_score_room_pd["room_type"],
    review_score_room_pd["avg_review_score"]
)
plt.title("Average Review Score by Room Type")
plt.xlabel("Room Type")
plt.ylabel("Average Review Score")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/review_score_by_room_type.png")
plt.show()


## Cell 37 - Monthly Review Trend


In [ ]:
monthly_review_trend = reviews_clean.where(col("date").isNotNull()) \
    .withColumn("review_month", date_format(col("date"), "yyyy-MM")) \
    .groupBy("review_month") \
    .agg(count("*").alias("review_count")) \
    .orderBy("review_month")

monthly_review_trend.show(30)


**Comment:** Monthly review trend provides a time-based view of market activity. Higher review counts may indicate higher booking activity, but review counts can also depend on guest behavior and data collection period.


## Cell 38 - Chart: Monthly Review Trend


In [ ]:
monthly_review_pd = monthly_review_trend.toPandas()

plt.figure(figsize=(12, 6))
plt.plot(
    monthly_review_pd["review_month"],
    monthly_review_pd["review_count"],
    marker="o"
)
plt.title("Monthly Review Trend")
plt.xlabel("Month")
plt.ylabel("Review Count")
plt.xticks(rotation=60)
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/monthly_review_trend.png")
plt.show()


## Cell 39 - Availability by Room Type


In [ ]:
listing_room_type = listings_clean.select(
    col("id").alias("listing_id_for_join"),
    "room_type"
)

calendar_listing_room = calendar_clean.join(
    listing_room_type,
    calendar_clean.listing_id == listing_room_type.listing_id_for_join,
    "inner"
)

availability_by_room_type = calendar_listing_room.groupBy("room_type") \
    .agg(
        count("*").alias("calendar_records"),
        round(avg("is_available"), 4).alias("availability_rate"),
        round(avg("calendar_price_clean"), 2).alias("avg_calendar_price")
    ) \
    .orderBy(desc("availability_rate"))

availability_by_room_type.show(truncate=False)


**Comment:** Availability by room type shows which accommodation types are more often available in the calendar data. It can indicate supply patterns and booking pressure by room type.


## Cell 40 - Chart: Availability by Room Type


In [ ]:
availability_room_pd = availability_by_room_type.toPandas()

plt.figure(figsize=(8, 6))
plt.bar(
    availability_room_pd["room_type"],
    availability_room_pd["availability_rate"]
)
plt.title("Availability Rate by Room Type")
plt.xlabel("Room Type")
plt.ylabel("Availability Rate")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/availability_by_room_type.png")
plt.show()


## Cell 41 - Top Expensive Neighbourhoods


In [ ]:
top_expensive_neighbourhoods = listings_clean.groupBy("neighbourhood_cleansed") \
    .agg(
        count("*").alias("listing_count"),
        round(avg("price_clean"), 2).alias("avg_price")
    ) \
    .where(col("listing_count") >= 10) \
    .orderBy(desc("avg_price")) \
    .limit(10)

top_expensive_neighbourhoods.show(truncate=False)


**Comment:** Top expensive neighbourhoods identify areas with the highest average listing prices. A minimum listing count filter is used to reduce distortion from neighbourhoods with only a few listings.


## Cell 42 - Chart: Top Expensive Neighbourhoods


In [ ]:
top_expensive_pd = top_expensive_neighbourhoods.toPandas()

plt.figure(figsize=(10, 6))
plt.barh(
    top_expensive_pd["neighbourhood_cleansed"],
    top_expensive_pd["avg_price"]
)
plt.title("Top 10 Expensive Neighbourhoods by Average Price")
plt.xlabel("Average Price")
plt.ylabel("Neighbourhood")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f"{OUTPUT_CHART_PATH}/top_expensive_neighbourhoods.png")
plt.show()


## Cell 43 - Save Output Tables


In [ ]:
host_listing_count.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/host_listing_count"
)

region_count.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/region_count"
)

room_type_count.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/room_type_count"
)

room_by_region.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/room_by_region"
)

avg_price_by_region_room.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/avg_price_by_region_room"
)

top_neighbourhoods.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/top_neighbourhoods"
)

minimum_nights_dist.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/minimum_nights_distribution"
)

popular_listings.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/popular_listings"
)

daily_availability.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/daily_availability"
)

listing_review_join.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/listing_review_join"
)

price_summary.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/price_summary"
)

price_quantile_summary.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/price_quantile_summary"
)

review_score_by_room_type.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/review_score_by_room_type"
)

monthly_review_trend.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/monthly_review_trend"
)

availability_by_room_type.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/availability_by_room_type"
)

top_expensive_neighbourhoods.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"{OUTPUT_TABLE_PATH}/top_expensive_neighbourhoods"
)

print("All output tables saved successfully.")




## Cell 44 - Verify Saved Output Files


In [ ]:
print("Charts:")
for f in os.listdir(OUTPUT_CHART_PATH):
    print("-", f)

print("\nTables:")
for f in os.listdir(OUTPUT_TABLE_PATH):
    print("-", f)


## Cell 45 - Final Analysis Summary


In [ ]:
print("Lab 4 Airbnb Team16 analysis completed.")
print("Main completed tasks:")
print("1. Connected to Spark cluster.")
print("2. Loaded 4 Airbnb datasets.")
print("3. Inspected schemas and sample records.")
print("4. Checked dataset geography and neighbourhood coverage.")
print("5. Cleaned price and checked missing values.")
print("6. Analyzed host concentration.")
print("7. Analyzed listing distribution by region.")
print("8. Analyzed listing distribution by room type.")
print("9. Analyzed room type distribution by region.")
print("10. Analyzed average price by region and room type.")
print("11. Analyzed top neighbourhoods.")
print("12. Analyzed minimum nights.")
print("13. Analyzed popular listings by reviews.")
print("14. Analyzed calendar date range and availability.")
print("15. Joined listings and reviews datasets.")
print("16. Visualized location and price distribution.")
print("17. Analyzed price distribution.")
print("18. Analyzed review score by room type.")
print("19. Analyzed monthly review trend.")
print("20. Analyzed availability by room type.")
print("21. Analyzed top expensive neighbourhoods.")
print("22. Saved output tables and charts.")

